<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Text Frequency - Solution

---

### About
This notebook explores **CountVectorizer** and **TF-IDF** through reviews of a product.  To use **CountVectorizer** and **TF-IDF** in a sentument analysis classifier, see the **Build a Sentiment Analysis Classifier Lab**.

### Learning Objective
- Apply CountVectorizer and TF-IDF to text.

### Notebook Guide
- NLP Scenario
- Introduction to CountVectorizer
- Stop words
- N-gram Length
- TF-IDF
- Try It!
- Conclusions and Takeaways

### Imports

In [ ]:
%pip install -qqq numpy pandas scikit-learn


In [ ]:
# imports
import numpy as np
import pandas as pd

# NLP imports
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# model building imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# code to avoid truncation of the output below 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# NLP Scenario 
You are an analyst for a marketing company that has just launched a new product suite of mobile devices. You have data from product reviews of one of these new products, the **TechWave X1**. For this exercise you will use the text from the reviews. 

#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"  **Star rating**: 5
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."  **Star rating**: 2
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."  **Star rating**: 5
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."  **Star rating**: 3
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."  **Star rating**: 1
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."  **Star rating**: 2

# Introduction to `CountVectorizer`
Computers understand the world in numbers. In order to use text data we have to first represent it in a way that computers can interpret. 

Let's convert the text above to a `pandas DataFrame` and start working with the text data. 

Note: this is an example with a very small data set for demo/walkthrough purposes, which is why we're manually converting it to a `pandas DataFrame`. In a real task (and in later tasks) you would import the data from csv or txt files. 

In [ ]:
# data setup
reviews_with_ratings = [
    ("I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!", 5),
    ("I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.", 2),
    ("The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.", 5),
    ("I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.", 3),
    ("The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.", 1),
    ("The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities.", 2)
]

# Create a DataFrame
df = pd.DataFrame(reviews_with_ratings, columns=['reviews', 'star_rating'])
df.head()

Next we will convert the text from the reviews into a matrix that creates a column for each word and tracks which reviews contain each word. Let's look at the output.

In [ ]:
# Initialize CountVectorizer for n-grams (single words in this case)
vectorizer = CountVectorizer(ngram_range=(1, 1))

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

# Stop words
Notice that we have some words that are so common (for example: "an", "and", "the") that they are not useful for our analysis. These words are called **stop words**. We can remove them by using the `stop_words` parameter in the `CountVectorizer` class. Notice the difference in the output below. 

In [ ]:
# Initialize CountVectorizer for n-grams (single words in this case), and remove stop_words
vectorizer = CountVectorizer(ngram_range=(1, 1), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

We went from 66 n-grams/words to 38 n-grams/words by removing the stop words. This will allow for more meaningful analysis of the reviews.  

# N-gram Length
Another modification we may want to make is adjusting how words are grouped. By using single words, we are missing important context such as "Techwave X1" being the product name of interest. Let's edit our code to capture bigrams. 

In [ ]:
# Initialize CountVectorizer for n-grams (single words in this case), remove stop_words, and look at bi-grams
vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

Interesting! Let's edit this a bit more to capture both bigrams and single words. 

In [ ]:
# Initialize CountVectorizer for n-grams (single words in this case), remove stop_words, and look at both single words and bi-grams
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

Great job! We've successfully created n-grams from the reviews using **CountVectorizer**. You've also learned how to filter out stop words and create n-grams of different lengths. This will be useful when you start using machine learning models to analyze text data. 

# TF-IDF

Now let's explore another popular technique for text analysis: **TF-IDF (Term Frequency-Inverse Document Frequency)**. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It is often used in information retrieval and text mining. Let's see how it works in practice.

TF-IDF combines two metrics:
- Term Frequency (TF): How often a word appears in a document
- Inverse Document Frequency (IDF): How unique that word is across all documents
This helps identify words that are both frequent and meaningful in specific documents.

We will first explore this with the social media posts we saw in the slides: 

- I love learning about text. 
- Text models are fun. 
- I love learning at GA. Learning is fun!

In [ ]:
# Sample text for demo 
text= ["I love learning about text.", "Text models are fun.", "I love learning at GA. Learning is fun!"]

Let's start by inspecting the TF-IDF of single words.

In [ ]:
# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 1))

# Fit and transform the reviews
X = vectorizer.fit_transform(text)

# Create a new DataFrame with the TF-IDF values
tfidf_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("tfidf_df shape:", tfidf_df.shape)
print("tfidf_df columns:", tfidf_df.columns)

# Display the new DataFrame
tfidf_df.head()

Now let's look at both single words (n-gram of 1) and bi-grams.

In [ ]:
# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the TF-IDF values
tfidf_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("tfidf_df shape:", tfidf_df.shape)
print("tfidf_df columns:", tfidf_df.columns)

# Display the new DataFrame
tfidf_df.head()

Notice that the TF-IDF values are different from the count values. This is because the TF-IDF values are normalized by the term frequency and inverse document frequency, which gives more weight to terms that are rare in the corpus and less weight to terms that are common. This helps to identify the most important terms in a document.

# Conclusions and Takeaways
- Bag-of-words methods represent text numerically so that it can be used for by the computer for analysis.
- These methods are effective while also being fast and cost effective to implement.
- These methods don't capture context from the words - if that is required for your task consider using more embeddings instead.